# Module B — Sentiment Analysis & Chatbot Intent Model Training
This notebook trains:
1. TF-IDF + Logistic Regression Sentiment Model on `reviews.csv`
2. TF-IDF + Multinomial Naive Bayes Intent Classifier on `intents.json`

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# 1. Sentiment Model Training
df = pd.read_csv('../data/reviews.csv')
print(df['sentiment'].value_counts())

vec = TfidfVectorizer(ngram_range=(1, 2))
X = vec.fit_transform(df['review_text'])
y = df['sentiment']

sentiment_model = LogisticRegression()
sentiment_model.fit(X, y)
print('Sentiment Model Accuracy:', sentiment_model.score(X, y))

# Save Sentiment Model
os.makedirs('../app/models', exist_ok=True)
joblib.dump(sentiment_model, '../app/models/sentiment_model.pkl')
joblib.dump(vec, '../app/models/sentiment_vectorizer.pkl')

In [ ]:
# 2. Chatbot Intent Model Training
with open('../data/intents.json', 'r') as f:
    data = json.load(f)

patterns, labels = [], []
for intent in data['intents']:
    for p in intent['patterns']:
        patterns.append(p)
        labels.append(intent['tag'])

cb_vec = TfidfVectorizer(ngram_range=(1, 2))
X_cb = cb_vec.fit_transform(patterns)
cb_model = MultinomialNB(alpha=0.1)
cb_model.fit(X_cb, labels)
print(f'Chatbot trained on {len(patterns)} pattern samples across {len(set(labels))} intents.')

joblib.dump(cb_model, '../app/models/chatbot_model.pkl')
joblib.dump(cb_vec, '../app/models/chatbot_vectorizer.pkl')